<div align="center">

###  Data Cleaning

**Cleaning and preparing the scraped product data for accurate analysis.**

**Workflow:** Load Data → Inspect Data → Clean Columns → Handle Missing Values → Remove Duplicates → Convert Data Types → Clean Text & Numbers → Validate Data → Save Cleaned Data

</div>


<div align="center">  

###  Libraries

**Python libraries used for data cleaning and analysis.**

</div>


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import re

# ============================================================
# LOAD DATA
# ============================================================

notebook_dir = Path.cwd()
file_candidates = [
    notebook_dir / "shopsy_kitchen_products.csv",
    notebook_dir / "notebooks/shopsy_kitchen_products.csv",
    notebook_dir.parent / "notebooks/shopsy_kitchen_products.csv",
    Path("notebooks/shopsy_kitchen_products.csv")
]
file_path = next((path for path in file_candidates if path.exists()), None)

if file_path is None:
    raise FileNotFoundError("Raw CSV not found in the notebooks folder.")

df = pd.read_csv(file_path)

print("Before Cleaning:", df.shape)
display(df.head())

Before Cleaning: (29, 9)


,Product_Name,Price,Discount,Rating,Reviews,Brand,Capacity,Material,Product_URL
0,Qtrix Pack of 8 Plastic Grocery Container - 50...,₹300,69%,4.4,104.0,Shopsy,"500 ml, 1100 ml, 1500 ml",Plastic,https://www.shopsy.in/qtrix-pack-8-plastic-gro...
1,AneriDEALS Pack of 24 Plastic Grocery Containe...,₹423,78%,4.1,34.0,Shopsy,"250 ml, 350 ml, 650 ml, 1200 ml, 1000 ml",Plastic,https://www.shopsy.in/anerideals-pack-24-plast...
2,BELIZZI Pack of 6 Plastic Fridge Container - 1...,₹257,74%,4.1,272.0,Shopsy,"1500 ml, 500 ml, 1000 ml",Plastic,https://www.shopsy.in/belizzi-pack-6-plastic-f...
3,SHAN Pack of 2 Ceramic Pickle Jar - 500 ml Green,₹345,50%,4.3,NaN,Shopsy,500 ml,Ceramic,https://www.shopsy.in/shan-pack-2-ceramic-pick...
4,"RK Pack of 5 Steel Cookie Jar - 300 ml, 500 ml...",₹383,76%,4.1,83.0,Shopsy,"300 ml, 500 ml, 750 ml, 1150 ml, 1650 ml, 375 ...",Steel,https://www.shopsy.in/rk-pack-5-steel-cookie-j...


<div align="center">

###  1. Clean Product Name

**Cleaning and standardizing product names for accurate analysis.**

</div>

In [ ]:


df["Product_Name"] = (
    df["Product_Name"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

<div align="center">

###  2. Clean Price

**Cleaning and converting product prices into a consistent numeric format for accurate analysis.**

</div>

In [ ]:


df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace(r"[₹,\s]", "", regex=True)
)

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

<div align="center">

###  3. Clean Discount

**Cleaning and converting discount values into a consistent numeric format for accurate analysis.**

</div>

In [ ]:


df["Discount"] = (
    df["Discount"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)

df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")

<div align="center">

### 4. Clean Rating

**Cleaning and converting product ratings into a consistent numeric format for accurate analysis.**

</div>

In [ ]:

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df.loc[~df["Rating"].between(0, 5), "Rating"] = np.nan

<div align="center">

###  5. Clean Reviews

**Cleaning and converting review counts into a consistent numeric format for accurate analysis.**

</div>

In [ ]:


df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Reviews"] = df["Reviews"].round().astype("Int64")

<div align="center">

###  6. Fix Brand

**Cleaning and standardizing brand names for consistent analysis.**

</div>

In [ ]:


def extract_brand(product_name):
    if pd.isna(product_name):
        return ""

    product_name = str(product_name).strip()
    match = re.match(
        r"^(.*?)\s+Pack\s+of\b",
        product_name,
        flags=re.I
    )

    if match:
        return match.group(1).strip()

    return ""


df["Brand"] = df["Product_Name"].apply(extract_brand)

<div align="center">

###  7. Clean Capacity

**Cleaning and standardizing product capacity values for accurate analysis.**

</div>

In [ ]:


df["Capacity"] = (
    df["Capacity"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


def clean_capacity(value):
    if not value:
        return ""

    parts = re.split(r",\s*", value)
    cleaned = []

    for part in parts:
        part = part.strip()
        if part and part.lower() not in [x.lower() for x in cleaned]:
            cleaned.append(part)

    return ", ".join(cleaned)


df["Capacity"] = df["Capacity"].apply(clean_capacity)

<div align="center">

###  8. Clean Material

**Cleaning and standardizing material values for consistent analysis.**

</div>

In [ ]:


df["Material"] = (
    df["Material"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

<div align="center">

###  9. Clean Product URL

**Cleaning and standardizing product URLs for reliable access and analysis.**

</div>

In [ ]:


df["Product_URL"] = (
    df["Product_URL"]
    .astype(str)
    .str.strip()
)

<div align="center">

###  10. Remove Duplicate Products

**Removing duplicate product records to maintain clean and accurate data.**

</div>

In [ ]:


df = df.drop_duplicates(subset=["Product_URL"])

<div align="center">

###  11. Handle Empty Values

**Identifying and handling missing or empty values to improve data quality.**

</div>

In [ ]:


df = df.replace(
    ["", "nan", "NaN", "None"],
    np.nan
)

<div align="center">

### 12. Final Column Order

**Organizing columns into a clear and consistent order for analysis.**

</div>

In [ ]:


df = df[
    [
        "Product_Name",
        "Price",
        "Discount",
        "Rating",
        "Reviews",
        "Brand",
        "Capacity",
        "Material",
        "Product_URL"
    ]
]

<div align="center">

###  13. Save Clean CSV

**Saving the cleaned product data as a CSV file for further analysis.**

</div>

In [ ]:


project_dir = notebook_dir if (notebook_dir / "Data").exists() else notebook_dir.parent
output_dir = project_dir / "Data" / "shopsy_db"
output_dir.mkdir(parents=True, exist_ok=True)
clean_file = output_dir / "shopsy_kitchen_products_cleaned.csv"

df.to_csv(
    clean_file,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned CSV saved successfully!")
print(clean_file)

Cleaned CSV saved successfully!
d:\DS\Data engineering courses\shopsy_home_kitchen_project\Data\shopsy_db\shopsy_kitchen_products_cleaned.csv


<div align="center">

###  14. Final Output

**Displaying the final cleaned dataset after completing all data cleaning steps.**

</div>

In [ ]:


print("\nAfter Cleaning:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nCleaned Data:")
display(df.head(10))

print("\nSaved:")
print(clean_file)


After Cleaning: (29, 9)

Missing Values:
Product_Name    0
Price           0
Discount        0
Rating          0
Reviews         3
Brand           0
Capacity        1
Material        0
Product_URL     0
dtype: int64

Cleaned Data:


,Product_Name,Price,Discount,Rating,Reviews,Brand,Capacity,Material,Product_URL
0,Qtrix Pack of 8 Plastic Grocery Container - 50...,300,69,4.4,104,Qtrix,"500 ml, 1100 ml, 1500 ml",Plastic,https://www.shopsy.in/qtrix-pack-8-plastic-gro...
1,AneriDEALS Pack of 24 Plastic Grocery Containe...,423,78,4.1,34,AneriDEALS,"250 ml, 350 ml, 650 ml, 1200 ml, 1000 ml",Plastic,https://www.shopsy.in/anerideals-pack-24-plast...
2,BELIZZI Pack of 6 Plastic Fridge Container - 1...,257,74,4.1,272,BELIZZI,"1500 ml, 500 ml, 1000 ml",Plastic,https://www.shopsy.in/belizzi-pack-6-plastic-f...
3,SHAN Pack of 2 Ceramic Pickle Jar - 500 ml Green,345,50,4.3,<NA>,SHAN,500 ml,Ceramic,https://www.shopsy.in/shan-pack-2-ceramic-pick...
4,"RK Pack of 5 Steel Cookie Jar - 300 ml, 500 ml...",383,76,4.1,83,RK,"300 ml, 500 ml, 750 ml, 1150 ml, 1650 ml, 375 ...",Steel,https://www.shopsy.in/rk-pack-5-steel-cookie-j...
5,VR Pack of 3 Plastic Grocery Container - 4500 ...,247,75,4.1,210,VR,"4500 ml, 250 ml",Plastic,https://www.shopsy.in/vr-pack-3-plastic-grocer...
6,Loknath Pack of 12 Plastic Utility Container -...,567,43,4.1,3,Loknath,"2000 ml, 1350 ml, 4000 ml","Plastic, Steel",https://www.shopsy.in/loknath-pack-12-plastic-...
7,Hoatzin Pack of 6 Plastic Grocery Container - ...,497,50,4.3,7,Hoatzin,"12000 ml, 8000 ml, 6000 ml, 3000 ml, 2000 ml, ...",Plastic,https://www.shopsy.in/hoatzin-pack-6-plastic-g...
8,Veksin Pack of 6 Plastic Grocery Container - 1...,497,64,4.3,23,Veksin,"1000 ml, 2000 ml, 3000 ml, 6000 ml, 8000 ml, 1...",Plastic,https://www.shopsy.in/veksin-pack-6-plastic-gr...
9,TASTIC Pack of 1 Plastic Grocery Container - 1...,125,74,3.5,0,TASTIC,1800 ml,Plastic,https://www.shopsy.in/tastic-pack-1-plastic-gr...



Saved:
d:\DS\Data engineering courses\shopsy_home_kitchen_project\Data\shopsy_db\shopsy_kitchen_products_cleaned.csv


In [16]:
print("Rows:", len(df))
print("Average price:", round(df["Price"].mean(), 2))
print("Average rating:", round(df["Rating"].mean(), 2))

Rows: 29
Average price: 333.69
Average rating: 4.1


<div align="center">

###  Conclusion

**The scraped product data was successfully cleaned, standardized, and validated.**

**The final dataset is ready for analysis, visualization, SQL queries, and dashboard development.**

</div>